# Notebook 05 — Interpretability & SHAP (RQ4)
**RQ4:** Does using population-informed features or retraining on elderly data reduce the gap,
and which features account for the improvement?

Methods:
- SHAP feature importance (Lundberg & Lee, 2017) on Random Forest and XGBoost
- Feature importance comparison: young-trained vs elderly-retrained models
- Per-activity false-alarm analysis on elderly ADLs
- Effect of elderly-specific threshold (RQ4 adaptation)


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/fall-detection-generalization'
    %pip install -q numpy pandas scipy scikit-learn xgboost shap matplotlib seaborn
    sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    REPO_ROOT = str(pathlib.Path('..').resolve())

DATA_ROOT = REPO_ROOT + '/data/raw'
RESULTS   = REPO_ROOT + '/results'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from sklearn.base import clone
from sklearn.metrics import f1_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.data.sisfall import load_sisfall
from src.models.classical import build_classical_models
from src.models.threshold import ThresholdDetector
from src.evaluation.metrics import compute_metrics

RESULTS_DIR = Path(RESULTS)
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
df_feat = pd.read_parquet(f'{RESULTS}/features_filtered.parquet')
meta_cols = ['label', 'subject_id', 'age_group']
feat_cols = [c for c in df_feat.columns if c not in meta_cols]

young_mask   = df_feat['age_group'] == 'young'
elderly_mask = df_feat['age_group'] == 'elderly'

X_young = df_feat.loc[young_mask,   feat_cols].values.astype(np.float32)
y_young = df_feat.loc[young_mask,   'label'].values
X_eld   = df_feat.loc[elderly_mask, feat_cols].values.astype(np.float32)
y_eld   = df_feat.loc[elderly_mask, 'label'].values

print(f'Young: {X_young.shape}  Elderly: {X_eld.shape}')

## 1. Train young-calibrated models and compute SHAP on elderly test set

In [ ]:
models = build_classical_models(random_state=42)

# Train on all young data
rf_young = clone(models['random_forest'])
rf_young.fit(X_young, y_young)

xgb_young = clone(models['xgboost'])
xgb_young.fit(X_young, y_young)

print('Young-calibrated models trained.')

In [ ]:
# SHAP — Random Forest on elderly test
rf_clf = rf_young.named_steps['clf']
scaler = rf_young.named_steps['scaler']
X_eld_scaled = scaler.transform(X_eld)

explainer_rf = shap.TreeExplainer(rf_clf)
shap_values_rf = explainer_rf.shap_values(X_eld_scaled)  # list of [n_samples, n_features] per class

# Fall class (index 1)
sv_fall = shap_values_rf[1] if isinstance(shap_values_rf, list) else shap_values_rf

print('SHAP values computed for RF (elderly test set).')

In [ ]:
# Summary plot — top 20 features
plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv_fall, X_eld_scaled,
    feature_names=feat_cols,
    max_display=20,
    show=False,
)
plt.title('SHAP feature importance — RF (young-trained) on elderly data')
plt.tight_layout()
plt.savefig(f'{RESULTS}/shap_rf_elderly.png', bbox_inches='tight')
plt.show()

In [ ]:
# XGBoost SHAP
xgb_clf = xgb_young.named_steps['clf']
X_eld_scaled_xgb = xgb_young.named_steps['scaler'].transform(X_eld)

explainer_xgb = shap.TreeExplainer(xgb_clf)
shap_values_xgb = explainer_xgb.shap_values(X_eld_scaled_xgb)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_xgb, X_eld_scaled_xgb, feature_names=feat_cols, max_display=20, show=False)
plt.title('SHAP feature importance — XGBoost (young-trained) on elderly data')
plt.tight_layout()
plt.savefig(f'{RESULTS}/shap_xgb_elderly.png', bbox_inches='tight')
plt.show()

## 2. RQ4 — Retraining on elderly data: does it reduce the gap?

In [ ]:
# Fine-tune on elderly data (if any falls exist in elderly set)
if y_eld.sum() > 0:
    rf_elderly = clone(models['random_forest'])
    rf_elderly.fit(X_eld, y_eld)

    # Compare on elderly held-out (use last 20% as proxy — proper CV in production)
    n = len(X_eld)
    cut = int(0.8 * n)
    X_tr, y_tr = X_eld[:cut], y_eld[:cut]
    X_te, y_te = X_eld[cut:], y_eld[cut:]

    rf_young_retrain = clone(models['random_forest'])
    rf_young_retrain.fit(X_tr, y_tr)  # train only on elderly
    y_pred_eld_retrained = rf_young_retrain.predict(X_te)
    y_pred_young_on_eld  = rf_young.predict(X_te)

    m_young_on_eld  = compute_metrics(y_te, y_pred_young_on_eld)
    m_eld_retrained = compute_metrics(y_te, y_pred_eld_retrained)

    comp = pd.DataFrame([
        {'model': 'young-trained',      **{k: m_young_on_eld[k]  for k in ('sensitivity','specificity','f1')}},
        {'model': 'elderly-retrained',  **{k: m_eld_retrained[k] for k in ('sensitivity','specificity','f1')}},
    ])
    print('\nRQ4 — retraining effect:')
    print(comp.to_string(index=False))
    comp.to_csv(f'{RESULTS}/rq4_retraining_comparison.csv', index=False)
else:
    print('No fall samples in elderly subset — retraining comparison not applicable.')
    print('(SisFall elderly subjects only performed ADLs; use Laavanya dataset for RQ4.)')

## 3. Per-activity false-alarm analysis on elderly ADLs

In [ ]:
# Load metadata to get activity labels for elderly ADL windows
meta_csv = Path(RESULTS) / 'sisfall_metadata.csv'
if meta_csv.exists():
    df_meta = pd.read_csv(meta_csv)
    # Match elderly ADL windows (approximate — windows inherit trial activity)
    df_elderly_adl = df_feat[elderly_mask & (df_feat['label'] == 0)].copy()

    # Predict with young-trained RF
    y_pred_adl = rf_young.predict(X_eld[y_eld == 0])
    df_elderly_adl = df_elderly_adl.reset_index(drop=True)
    df_elderly_adl['predicted_fall'] = y_pred_adl

    # False-alarm rate is tricky without per-window activity labels
    # Here we just report overall and per-subject false-alarm rates
    far = y_pred_adl.mean()
    print(f'False alarm rate on elderly ADLs: {far:.3f} ({far*100:.1f}%)')
    print(f'(Windows incorrectly classified as falls: {y_pred_adl.sum()} / {len(y_pred_adl)})')
else:
    print('Metadata CSV not found — run Notebook 01 first.')

## 4. SHAP feature importance — top 10 feature bar chart (for thesis figure)

In [ ]:
mean_abs_shap = np.abs(sv_fall).mean(axis=0)
shap_df = pd.DataFrame({'feature': feat_cols, 'mean_abs_shap': mean_abs_shap})
shap_df = shap_df.sort_values('mean_abs_shap', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(shap_df['feature'][::-1], shap_df['mean_abs_shap'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top 15 features — RF (young-trained) on elderly test')
plt.tight_layout()
plt.savefig(f'{RESULTS}/shap_top15_bar.png', bbox_inches='tight')
plt.show()

shap_df.to_csv(f'{RESULTS}/shap_feature_importance.csv', index=False)